In [1]:
import json
import re
from pathlib import Path
from typing import List, Tuple, Optional
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

In [2]:
fs = sorted(Path("../data/json2").glob("*.json"))
print(f"{len(fs)} threads")

10257 threads


In [3]:
from IPython.display import display
# load output from
df3 = pd.read_parquet("../outputs/df_links4.parquet")
df3

,score,n_links,n_comments,comments,thread_urls,first_link_utc,last_link_utc,url
title,,,,,,,,
Worth the Candle,1445,108,108,[{'author_flair_text': 'Self-Appointed Court S...,[https://reddit.com/r/rational/comments/zvov5c...,2017-07-28 13:17:36,2024-04-17 19:03:01,[https://archiveofourown.org/works/11478249/ch...
"Alexander Wales - The Metropolitan Man, Shadows of the Limelight",1408,22,22,[{'author_flair_text': 'Time flies like an arr...,[https://reddit.com/r/rational/comments/zvov5c...,2015-04-18 18:13:47,2021-04-29 20:00:04,[https://www.patreon.com/alexanderwales]
Reddit - Dive into anything,1330,72,72,"[{'author_flair_text': 'Team Glimglam', 'body'...",[https://reddit.com/r/rational/comments/zvov5c...,2014-09-04 21:14:23,2024-07-31 01:19:04,[https://reddit.com/r/motheroflearning/comment...
Mother of Learning,859,84,84,"[{'author_flair_text': 'Utopian Smut Peddler',...",[https://reddit.com/r/rational/comments/zvov5c...,2014-07-26 08:21:12,2024-05-21 07:04:42,[https://www.fictionpress.com/s/2961893/1/Moth...
The Metropolitan Man,801,58,58,[{'author_flair_text': 'Time flies like an arr...,[https://reddit.com/r/rational/comments/zvov5c...,2014-05-31 03:39:09,2023-06-26 19:21:24,[https://www.fanfiction.net/s/10360716/1/The-M...
...,...,...,...,...,...,...,...,...
I know Bouletcorp did one...,16,2,2,[{'author_flair_text': 'Time flies like an arr...,[https://reddit.com/r/rational/comments/zvov5c...,2015-04-30 04:07:01,2015-10-29 01:28:38,[http://english.bouletcorp.com/2014/09/05/king...
The Writing on the Wall,16,3,3,"[{'author_flair_text': None, 'body': '[CORDYCE...",[https://reddit.com/r/rational/comments/zvov5c...,2016-06-22 23:40:41,2019-10-31 18:48:58,[https://www.fimfiction.net/story/42409/the-wr...
The actual arguments I heard against trusting fact checkers,16,2,2,"[{'author_flair_text': None, 'body': 'The best...",[https://reddit.com/r/rational/comments/zvov5c...,2016-11-28 18:49:35,2017-01-16 19:08:14,[https://www.reddit.com/r/AskTrumpSupporters/c...


## Extra get a llm summary of each link [WIP]

Grab all md's that mention a story, ask claude to summarize

We could also get total karma per mention

In [4]:
import dotenv
from anycache import anycache

dotenv.load_dotenv()
from openai import OpenAI

client = OpenAI()

import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")
cost = 0.150 / 1e6

In [41]:
# load all md posts 
md_posts = []
fs = sorted(Path("../data/cache2").glob("*.md"))
for f in fs:
    s = f.open().read()
    md_posts.append(s)


# order by date
def md2date(s: str) -> str:
    return s.split('* Created: ')[1].split('\n')[0]

md_posts = sorted(md_posts, key=md2date)
md2date(md_posts[0]), md2date(md_posts[-1])

('2009-11-25T02:34:03', '2024-12-16T15:00:13')

In [45]:

def get_post_context(urls: List[str], char_budget=100000, min_size=1000) -> str:
    # TODO maybe I should just get comment with links, and children?
    assert len(urls) > 0
    # from comments
    # return df3.loc[title].comments

    # or I could just all markdowns with

    # TODO use langchain chunking?

    matches = []
    for ii, post in enumerate(md_posts):
        for url in urls:
            if url in post:
                matches.append(post)
                break

    budget_pp = char_budget / len(matches)
    budget_pp = max(budget_pp, min_size)
    s = ""
    for i in range(len(matches)):
        post = matches[i]

        for url in urls:
            if url in post:
                ind = post.index(url)

        i0 = int(max(0, ind - budget_pp // 4))
        i1 = int(min(len(post), ind + budget_pp // 4 * 3))
        post_chunk = post[i0:i1]
        if i0 > 0:
            post_chunk = "..." + post_chunk
        if i1 < len(post):
            post_chunk = post_chunk + "..."

        s += f"\n\n----- Thread {ii} -----\n\n" + post_chunk

    # if too large get first N//2 and last N//2
    if len(s) > char_budget:
        s = s[:char_budget // 2] + "..." + s[-char_budget // 2:]
    return s


# url = df3.url[0].split('\n')
# print(url)
# c = get_context(url, 400000)
# print(c[:1000])

In [46]:
df3.iloc[-1].url

array(['https://www.amazon.com/Becoming-Batman-Possibility-Paul-Zehr/dp/0801890632'],
      dtype=object)

In [49]:
# QC test with long and short context
# urls = ['https://archiveofourown.org/works/11478249/chapters/25740126',
#        'https://www.royalroad.com/fiction/25137/worth-the-candle',
#        'http://archiveofourown.org/works/11478249?view_full_work=true',
#        'http://archiveofourown.org/works/11478249/chapters/25740126',
#        'https://archiveofourown.org/works/11478249']
# c = get_post_context(urls, 400000)
# print(urls)
# print(c)

# # urls = ['https://www.amazon.com/Becoming-Batman-Possibility-Paul-Zehr/dp/0801890632']
# # c = get_post_context(urls, 400000)
# # print(urls)
# # print(c)


In [117]:


from pydantic import BaseModel, Field


class FictionInfo(BaseModel):
    title: str
    description: str = Field(description="A few paragraphs of very concise, informative, dense, description of the work")
    tags: List[str] = Field(
        description="""Long list of common descriptors: format (web serial, fanfic, lightnovel, short, complete, comic), genre (scifi, fantasy)
        Key elements (rational, timeloop, litrpg, progression, cultivation, isekai)
        Content notes (grimdark, romance, harem, queer, funny, NSFW)
        """
    )

    # status: Optional[str] = Field(description="complete/ongoing/hiatus/abandoned")
    # type: str = Field(description='e.g. fanfiction, original, comic, etc.')

    reviews_quotes: List[str] = Field(
        description="Directly and fully quote excerpts from every single users' comments about the fiction"
    )
    reviews_summary: Optional[str] = Field(
        description="Structured, dry, and concise summary of reviews"
    )
    reccomendations: Optional[str] = Field(
        description="Why users recommend it"
    )
    disrecommendations: Optional[str] = Field(
        description="Why users disrecommend it"
    )
    why: Optional[str] = Field(
        description="Why/when might readers of r/rational like it"
    )
    if_you_liked_x_you_will_like_this: List[str] = Field(
        description="Fans of X will also like the work. Exhaustively list ALL of X mentioned in the discussions."
    )

    quality: float = Field(
        # ge=0.0, le=10.0,
        description="Overall user sentiment out of 10"
    )
    rationality: Optional[float] = Field(
        # ge=0.0, le=10.0,
        description="Systematic worldbuilding, character competence, logical consistency. Where HPMOR is a 10 and Worm is a 5."
    )
    rating_writing: Optional[float]
    rating_plot: Optional[float]
    rating_character: Optional[float]
    rating_worldbuilding: Optional[float]


f_cache_llm = Path("../outputs/.anycache3")


# @anycache(f_cache_llm)
def get_llm_summary(title: str, urls: str, context: str):
    chat_completion = client.beta.chat.completions.parse(
        messages=[
            {
                "role": "system",
                "content": "You are Gwern Branwern, an internet librarian who specializes in rational fiction. You are summarising community reccomendations from r/rational into a dry, informative, concise, and structured format for your own personal notes. Because it's private you can be consise, frank, and opinionated.",
            },
            {
                "role": "user",
                "content": f"""Using the given structure, summarize the parts of these discussions where they talk about {title} (urls: {urls}).

### Context:

{context}""",
            },
        ],
        model="gpt-4o-mini",
        response_format=FictionInfo,
    )

    return chat_completion.choices[0].message.parsed.__dict__

In [118]:
# import shutil
# shutil.rmtree(f_cache_llm, ignore_errors=True)

In [119]:
from openai.lib._pydantic import to_strict_json_schema

to_strict_json_schema(FictionInfo)

{'properties': {'title': {'title': 'Title', 'type': 'string'},
  'description': {'description': 'A few paragraphs of very concise, informative, dense, description of the work',
   'title': 'Description',
   'type': 'string'},
  'tags': {'description': 'Long list of common descriptors: format (web serial, fanfic, lightnovel, short, complete, comic), genre (scifi, fantasy)\n        Key elements (rational, timeloop, litrpg, progression, cultivation, isekai)\n        Content notes (grimdark, romance, harem, queer, funny, NSFW)\n        ',
   'items': {'type': 'string'},
   'title': 'Tags',
   'type': 'array'},
  'reviews_quotes': {'description': "Directly and fully quote excerpts from every single users' comments about the fiction",
   'items': {'type': 'string'},
   'title': 'Reviews Quotes',
   'type': 'array'},
  'reviews_summary': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'description': 'Structured, dry, and concise summary of reviews',
   'title': 'Reviews Summary'},
  'rec

In [120]:
llm_info = []


l = min(1000, len(df3))
for i in tqdm(range(0, l)):
    title = df3.index[i]
    urls = df3.iloc[i].url

    context = get_post_context(urls, char_budget=50000)
    tokens = len(enc.encode(context))
    print(
        f"Input Tokens: {tokens}. Input Cost: {cost * tokens:.4f} USD, for title={title} with urls={urls}"
    )

    llm_data = get_llm_summary(title, urls, context)

    llm_data["title2"] = title
    llm_data["url"] = urls

    if i==0:
        print(f"Content: {context[:1000]}")
        display(llm_data)

    llm_info.append(llm_data)

    1/0

  0%|          | 0/1000 [00:00<?, ?it/s]

Input Tokens: 13825. Input Cost: 0.0021 USD, for title=Worth the Candle with urls=['https://archiveofourown.org/works/11478249/chapters/25740126'
 'https://www.royalroad.com/fiction/25137/worth-the-candle'
 'http://archiveofourown.org/works/11478249?view_full_work=true'
 'http://archiveofourown.org/works/11478249/chapters/25740126'
 'https://archiveofourown.org/works/11478249']
Content: 

----- Thread 10261 -----

## [RT][WIP] Worth the Candle, Ch 1

* Author: u/cthulhuraejepsen  *Fruit flies like a banana**
* URL: http://archiveofourown.org/works/11478249/chapters/25740126
* Score: 33

* Created: 2017-07-14T05:23:38

### Post:

[Link to content](http://archiveofourown.org/works/11478249/chapters/25740126)

### Comments:

> **u/cthulhuraejepsen** [+9]  *Fruit flies like a banana**
> 
> This is a self-insert litRPG portal fantasy, and I would have made it a Groundhog Day loop if I thought I could fit that in.
> 

> **u/LucidityWaver** [+8] *
> 
> > loosely based on my personal experienc

{'title': 'Worth the Candle',
 'description': "*Worth the Candle* is a self-insert litRPG portal fantasy novel written by Alexander Wales. The story follows Juniper, a teenage boy who unexpectedly finds himself in a fantasy realm reminiscent of various tabletop games he has played and run as a Dungeon Master. In this world, Juniper possesses a character sheet and must navigate the challenges and mechanics of a complex, game-like system while dealing with themes of agency, friendship, and the consequences of his actions. The work explores the protagonist's growth through various trials, character interactions, and the use of humor interspersed with serious topics such as trauma and morality, making it an engaging read for fans of the genre.",
 'tags': ['web serial',
  'litRPG',
  'fantasy',
  'self-insert',
  'portal fantasy',
  'grimdark',
  'humor',
  'progression'],
 'reviews_quotes': ['"This is quickly becoming my favorite litrpg. You need to work a little harder to catch up to \'Wa

ZeroDivisionError: division by zero

In [12]:
i

1

In [ ]:
df_llm = pd.DataFrame(llm_info)
df_llm
# also join with df4